In [1]:
import os
os.environ["OLLAMA_HOST"] = "http://localhost:11300" 
print(os.environ.get("OLLAMA_HOST"))

http://localhost:11300


In [ ]:
pip install -r requirements.txt

In [3]:
import Prompts.ver_3 as prompts
import importlib
importlib.reload(prompts)
from Prompts.ver_3 import Config
import evaluate
importlib.reload(evaluate)

<module 'evaluate' from '/home/aluc31or/LLM_Project/evaluate.py'>

In [4]:
import numpy as np
import pandas as pd
from langchain_ollama import OllamaLLM
from langchain_core.prompts import PromptTemplate
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

In [5]:
def evaluate_model(df, results):
    # Get true labels from the dataframe
    y_true = df["Binary_label"].tolist()
    y_pred = results

    # Calculate metrics
    accuracy = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='weighted', pos_label=None)
    conf_matrix = confusion_matrix(y_true, y_pred)

    return accuracy, f1, conf_matrix

In [6]:
def run_prompt_eval(dataset_path, prompt_template, model, text_col="text"):
    
    df_local = pd.read_csv(dataset_path)
    reviews = df_local[text_col].tolist()

    prompt = PromptTemplate(
        input_variables=["text"],
        template=prompt_template,
    )

    llm = OllamaLLM(model=model)
    chain = prompt | llm

    predictions = []
    for review in reviews:
        response = chain.invoke({"text": review})
        label = response.strip().lower()

        if "fake" in label:
            cleaned_label = "fake"
        elif "real" in label:
            cleaned_label = "real"
        else:
            cleaned_label = "fake"  # safety fallback
        
        cleaned_label = cleaned_label.strip('.,!?;:')

        #print(f"Raw output: {response} -> Cleaned: {cleaned_label}")
        predictions.append(cleaned_label)

    accuracy, f1, conf_matrix = evaluate_model(df_local, predictions)
    return df_local, predictions, accuracy, f1, conf_matrix

# 1. Amazon Human vs Computer

In [7]:
# Load CSV
df_reviews = pd.read_csv("/home/aluc31or/LLM_Project/Amazon_Human_VS_ComputerFake.csv")
reviews = df_reviews["text"].tolist()
df_reviews.head()

,Binary_label,Category,source,text,is_synthetic
0,fake,Home_and_Kitchen,amazon,Air pressure was utter crap. I kept the lid o...,1
1,fake,Home_and_Kitchen,amazon,"Not what I expected, smaller than expected, bu...",1
2,fake,Home_and_Kitchen,amazon,We like a toaster that doesn't have the long h...,1
3,fake,Home_and_Kitchen,amazon,Love these pans they work great. The only prob...,1
4,fake,Home_and_Kitchen,amazon,I love the print on the back and the finish. I...,1


## 1.1 Zero Shot Prompting

In [8]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Amazon_Human_VS_ComputerFake.csv",
    prompt_template=Config.zero_shot_depth0_width0,
    model="llama3:8b"
 )

In [9]:
print("=" * 50)
print("zero_shot_depth0_width0 Results")
print("Model: llama3:8b")
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_depth0_width0 Results
Model: llama3:8b
Accuracy: 0.5925

F1 Score: 0.5920691230111805

Confusion Matrix:
[[224 176]
 [150 250]]


In [10]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Amazon_Human_VS_ComputerFake.csv",
    prompt_template=Config.zero_shot_depth1_width1,
    model="llama3:8b"
 )

In [11]:
print("=" * 50)
print("zero_shot_depth1_width1 Results")
print("Model: llama3:8b")
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_depth1_width1 Results
Model: llama3:8b
Accuracy: 0.60375

F1 Score: 0.6026018920286892

Confusion Matrix:
[[220 180]
 [137 263]]


## 1.2 One-Shot Prompting

In [12]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Amazon_Human_VS_ComputerFake.csv",
    prompt_template=Config.one_shot_depth0_width0,
    model="llama3:8b"
 )

In [13]:
print("=" * 50)
print("one_shot_depth0_width0 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_depth0_width0 Results
Model: llama3:8b
Accuracy: 0.7525

F1 Score: 0.7502459698781503

Confusion Matrix:
[[339  61]
 [137 263]]


In [14]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Amazon_Human_VS_ComputerFake.csv",
    prompt_template=Config.one_shot_depth1_width1,
    model="llama3:8b"
 )

In [15]:
print("=" * 50)
print("one_shot_depth1_width1 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_depth1_width1 Results
Model: llama3:8b
Accuracy: 0.605

F1 Score: 0.6035728623042955

Confusion Matrix:
[[266 134]
 [182 218]]


## 1.3 Few-Shot Prompting

In [16]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Amazon_Human_VS_ComputerFake.csv",
    prompt_template=Config.few_shot_depth0_width0,
    model="llama3:8b"
 )

In [17]:
print("=" * 50)
print("few_shot_depth0_width0 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_depth0_width0 Results
Model: llama3:8b
Accuracy: 0.72875

F1 Score: 0.721039801642587

Confusion Matrix:
[[358  42]
 [175 225]]


In [18]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Amazon_Human_VS_ComputerFake.csv",
    prompt_template=Config.few_shot_depth1_width1,
    model="llama3:8b"
 )

In [19]:
print("=" * 50)
print("few_shot_depth1_width1 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_depth1_width1 Results
Model: llama3:8b
Accuracy: 0.6025

F1 Score: 0.5930355326051485

Confusion Matrix:
[[302  98]
 [220 180]]


# 2. Hotel Human Real vs Human Fake

In [20]:
from Prompts.ver_3 import Config
print(Config.zero_shot_depth0_width0)


    Role: You are an expert at classifying "real" and "fake" reviews.

    Task: Classify the following hotel or product review as either “real” or “fake.”

    Review
    "{text}"

    Guidelines:
    - Fake reviews often use generic, exaggerated, or marketing-style language.
    - Real reviews usually contain natural wording and specific details.

    Answer with your classification.
    Use one of these labels:
    "real" or "fake"
    


In [21]:
# Load CSV
df_reviews = pd.read_csv("/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake.csv")
reviews = df_reviews["text"].tolist()
df_reviews.head()

,Binary_label,Category,domain,text,is_synthetic,source
0,truthful,conrad,Hotel,We stayed for a one night getaway with family ...,0,TripAdvisor
1,truthful,hyatt,Hotel,Triple A rate with upgrade to view room was le...,0,TripAdvisor
2,truthful,hyatt,Hotel,This comes a little late as I'm finally catchi...,0,TripAdvisor
3,truthful,omni,Hotel,The Omni Chicago really delivers on all fronts...,0,TripAdvisor
4,truthful,hyatt,Hotel,I asked for a high floor away from the elevato...,0,TripAdvisor


## 2.1 Zero Shot Prompting

In [22]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake_relabelled.csv",
    prompt_template=Config.zero_shot_depth0_width0,
    model="llama3:8b"
 )

In [23]:
print("=" * 50)
print("zero_shot_depth0_width0 Results")
print("Model: llama3:8b")
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_depth0_width0 Results
Model: llama3:8b
Accuracy: 0.5571608040201005

F1 Score: 0.5328745429611332

Confusion Matrix:
[[262 534]
 [171 625]]


In [24]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake_relabelled.csv",
    prompt_template=Config.zero_shot_depth1_width1,
    model="llama3:8b"
 )

In [25]:
print("=" * 50)
print("zero_shot_depth1_width1 Results")
print("Model: llama3:8b")
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_depth1_width1 Results
Model: llama3:8b
Accuracy: 0.5383165829145728

F1 Score: 0.5192555387791575

Confusion Matrix:
[[270 526]
 [209 587]]


## 2.2 One-Shot Prompting

In [26]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake_relabelled.csv",
    prompt_template=Config.one_shot_depth0_width0,
    model="llama3:8b")

In [27]:
print("=" * 50)
print("one_shot_depth0_width0 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_depth0_width0 Results
Model: llama3:8b
Accuracy: 0.5986180904522613

F1 Score: 0.5575544778076572

Confusion Matrix:
[[234 562]
 [ 77 719]]


In [28]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake_relabelled.csv",
    prompt_template=Config.one_shot_depth1_width1,
    model="llama3:8b")

In [29]:
print("=" * 50)
print("one_shot_depth1_width1 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_depth1_width1 Results
Model: llama3:8b
Accuracy: 0.550251256281407

F1 Score: 0.5292835055633375

Confusion Matrix:
[[270 526]
 [190 606]]


## 2.3 Few-Shot Prompting

In [30]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake_relabelled.csv",
    prompt_template=Config.few_shot_depth0_width0,
    model="llama3:8b"
 )

In [31]:
print("=" * 50)
print("few_shot_depth0_width0 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_depth0_width0 Results
Model: llama3:8b
Accuracy: 0.6425879396984925

F1 Score: 0.6354347646730938

Confusion Matrix:
[[400 396]
 [173 623]]


In [32]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_Human_VS_HumanFake_relabelled.csv",
    prompt_template=Config.few_shot_depth1_width1,
    model="llama3:8b"
 )

In [33]:
print("=" * 50)
print("few_shot_depth1_width1 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_depth1_width1 Results
Model: llama3:8b
Accuracy: 0.550251256281407

F1 Score: 0.5371341324757172

Confusion Matrix:
[[304 492]
 [224 572]]


# 3. Human Real vs CG

In [34]:
from Prompts.ver_3 import Config
print(Config.zero_shot_depth0_width0)


    Role: You are an expert at classifying "real" and "fake" reviews.

    Task: Classify the following hotel or product review as either “real” or “fake.”

    Review
    "{text}"

    Guidelines:
    - Fake reviews often use generic, exaggerated, or marketing-style language.
    - Real reviews usually contain natural wording and specific details.

    Answer with your classification.
    Use one of these labels:
    "real" or "fake"
    


In [35]:
# Load CSV
df_reviews = pd.read_csv("/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv")
reviews = df_reviews["text"].tolist()
df_reviews.head()

,Binary_label,Category,domain,text,is_synthetic,source
0,real,james,Hotel,My husband and I were there for a conference a...,0,Web
1,real,palmer,Hotel,Here are my experiences: - Had three rooms res...,0,Web
2,real,omni,Hotel,My wife and I travelled to Chicago and really ...,0,TripAdvisor
3,real,knickerbocker,Hotel,The location of this hotel is excellent. The h...,0,Web
4,real,fairmont,Hotel,We recently completed our second stay at the F...,0,TripAdvisor


## 3.1 Zero Shot Prompting

In [36]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv",
    prompt_template=Config.zero_shot_depth0_width0,
    model="llama3:8b"
 )

In [37]:
print("=" * 50)
print("zero_shot_depth0_width0 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_depth0_width0 Results
Model: llama3:8b
Accuracy: 0.5163316582914573

F1 Score: 0.48545226506109057

Confusion Matrix:
[[216 580]
 [190 606]]


In [38]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv",
    prompt_template=Config.zero_shot_depth1_width1,
    model="llama3:8b"
 )

In [39]:
print("=" * 50)
print("zero_shot_depth1_width1 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_depth1_width1 Results
Model: llama3:8b
Accuracy: 0.37374371859296485

F1 Score: 0.27979130961925985

Confusion Matrix:
[[ 10 786]
 [211 585]]


## 3.2 One-Shot Prompting

In [40]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv",
    prompt_template=Config.one_shot_depth0_width0,
    model="llama3:8b"
 )

In [41]:
print("=" * 50)
print("one_shot_depth0_width0 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_depth0_width0 Results
Model: llama3:8b
Accuracy: 0.5835427135678392

F1 Score: 0.5369492230325092

Confusion Matrix:
[[212 584]
 [ 79 717]]


In [42]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv",
    prompt_template=Config.one_shot_depth1_width1,
    model="llama3:8b"
 )

In [43]:
print("=" * 50)
print("one_shot_depth1_width1 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_depth1_width1 Results
Model: llama3:8b
Accuracy: 0.4434673366834171

F1 Score: 0.3836381687501748

Confusion Matrix:
[[105 691]
 [195 601]]


## 3.3 Few-Shot Prompting

In [44]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv",
    prompt_template=Config.few_shot_depth0_width0,
    model="llama3:8b"
 )

In [45]:
print("=" * 50)
print("few_shot_depth0_width0 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_depth0_width0 Results
Model: llama3:8b
Accuracy: 0.6890703517587939

F1 Score: 0.6862477171499519

Confusion Matrix:
[[473 323]
 [172 624]]


In [46]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_CG.csv",
    prompt_template=Config.few_shot_depth1_width1,
    model="llama3:8b"
 )

In [47]:
print("=" * 50)
print("few_shot_depth1_width1 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_depth1_width1 Results
Model: llama3:8b
Accuracy: 0.457286432160804

F1 Score: 0.418733550713246

Confusion Matrix:
[[159 637]
 [227 569]]


# 4. Human Real vs MixFake

In [48]:
from Prompts.ver_3 import Config
print(Config.zero_shot_depth0_width0)


    Role: You are an expert at classifying "real" and "fake" reviews.

    Task: Classify the following hotel or product review as either “real” or “fake.”

    Review
    "{text}"

    Guidelines:
    - Fake reviews often use generic, exaggerated, or marketing-style language.
    - Real reviews usually contain natural wording and specific details.

    Answer with your classification.
    Use one of these labels:
    "real" or "fake"
    


In [49]:
# Load CSV
df_reviews = pd.read_csv("/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv")
reviews = df_reviews["text"].tolist()
df_reviews.head()

,Binary_label,Category,domain,text,is_synthetic,source
0,real,sofitel,Hotel,Just got back from a 10 day visit to Chicago. ...,0,TripAdvisor
1,real,palmer,Hotel,Just returned from a week in Chicago with the ...,0,TripAdvisor
2,real,hardrock,Hotel,Not worth the price. We almost stayed at a hot...,0,Web
3,real,homewood,Hotel,WARNING: FOOD POISONING ALERT!!! The room was ...,0,Web
4,real,ambassador,Hotel,Upon check in - the lobby was BUSY! I was tire...,0,TripAdvisor


## 4.1 Zero Shot Prompting

In [50]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv",
    prompt_template=Config.zero_shot_depth0_width0,
    model="llama3:8b"
 )

In [51]:
print("=" * 50)
print("zero_shot_depth0_width0 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_depth0_width0 Results
Model: llama3:8b
Accuracy: 0.5571608040201005

F1 Score: 0.5323061537019095

Confusion Matrix:
[[260 536]
 [169 627]]


In [52]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv",
    prompt_template=Config.zero_shot_depth1_width1,
    model="llama3:8b"
 )

In [53]:
print("=" * 50)
print("zero_shot_depth1_width1 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

zero_shot_depth1_width1 Results
Model: llama3:8b
Accuracy: 0.44974874371859297

F1 Score: 0.3996239125493356

Confusion Matrix:
[[128 668]
 [208 588]]


## 4.2 One-Shot Prompting

In [54]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv",
    prompt_template=Config.one_shot_depth0_width0,
    model="llama3:8b"
 )

In [55]:
print("=" * 50)
print("one_shot_depth0_width0 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_depth0_width0 Results
Model: llama3:8b
Accuracy: 0.6074120603015075

F1 Score: 0.5694010780119745

Confusion Matrix:
[[247 549]
 [ 76 720]]


In [56]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv",
    prompt_template=Config.one_shot_depth1_width1,
    model="llama3:8b"
 )

In [57]:
print("=" * 50)
print("one_shot_depth1_width1 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

one_shot_depth1_width1 Results
Model: llama3:8b
Accuracy: 0.4899497487437186

F1 Score: 0.45333739301058895

Confusion Matrix:
[[184 612]
 [200 596]]


## 4.3 Few-Shot Prompting

In [58]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv",
    prompt_template=Config.few_shot_depth0_width0,
    model="llama3:8b"
 )

In [59]:
print("=" * 50)
print("few_shot_depth0_width0 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_depth0_width0 Results
Model: llama3:8b
Accuracy: 0.6714824120603015

F1 Score: 0.6676479499030624

Confusion Matrix:
[[449 347]
 [176 620]]


In [60]:
df, results, accuracy, f1, conf_matrix = run_prompt_eval(
    dataset_path="/home/aluc31or/LLM_Project/Hotel_HumanReal_VS_MixFake.csv",
    prompt_template=Config.few_shot_depth1_width1,
    model="llama3:8b"
 )

In [61]:
print("=" * 50)
print("few_shot_depth1_width1 Results")
print("Model: llama3:8b")
print("=" * 50)
print("Accuracy:", accuracy)
print("\nF1 Score:", f1)
print("\nConfusion Matrix:")
print(conf_matrix)
print("=" * 50)

few_shot_depth1_width1 Results
Model: llama3:8b
Accuracy: 0.49685929648241206

F1 Score: 0.47176348509043914

Confusion Matrix:
[[222 574]
 [227 569]]
